In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# ====================================================================
# 🔹 BLOCK 1: SETUP, FUNCTIONS, AND INITIAL PARSING
# ====================================================================

# --- Step 1.1: Installs and Imports ---
!pip install -q pandas numpy scikit-learn sentence_transformers torch torchvision tqdm joblib
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
from torchvision import models, transforms
import torch
from PIL import Image
import requests
from io import BytesIO
from tqdm.auto import tqdm
import joblib
import os
import time

# --- Step 1.2: Path Definitions ---
BASE_PATH = '/content/drive/My Drive/ML_Price_Prediction/dataset/'
MODELS_PATH = '/content/drive/My Drive/ML_Price_Prediction/models/'
os.makedirs(MODELS_PATH, exist_ok=True)

# --- Define paths for intermediate and final files ---
TRAIN_PARSED_PATH = os.path.join(BASE_PATH, 'train_parsed.csv')
IMG_EMBED_CHECKPOINT_PATH = os.path.join(BASE_PATH, 'train_image_embeddings_v2_checkpoint.csv')
TEXT_EMBED_CHECKPOINT_PATH = os.path.join(BASE_PATH, 'train_text_embeddings_v2_checkpoint.csv')
TRAIN_FINAL_CLEANED_PATH = os.path.join(BASE_PATH, 'train_final_cleaned_v2.csv')
SCALER_PATH = os.path.join(MODELS_PATH, 'scaler_v2.pkl')
OUTLIER_CAPS_PATH = os.path.join(MODELS_PATH, 'outlier_caps_v2.npy')

# --- Step 1.3: Function Definitions ---
def parse_product_details_v5(text):
    num_of_packs, net_qty_value, unit = 1, 1.0, 'count'
    text_lower = str(text).lower()
    pack_patterns = [
        r'\(pack of (\d+)\)', r'pack of (\d+)', r'(\d+)\s*per case', r'(\d+)\s*count',
        r'(\d+)\s*ct', r'(\d+)\s*pack', r'(\d+)-pack', r'(\d+)-ct', r'(\d+)\s*pcs'
    ]
    found_packs = [1]
    for pattern in pack_patterns:
        matches = re.findall(pattern, text_lower)
        if matches:
            for match in matches:
                found_packs.append(int(match))
    num_of_packs = max(found_packs)
    qty_pattern = r'(\d+\.?\d*)\s*-?\s*(fluid ounce|fl oz|ounces|ounce|oz|grams|gram|g|pounds|pound|lbs|lb|liters|liter|l|milliliters|milliliter|ml)\b'
    qty_match = re.search(qty_pattern, text_lower)
    if qty_match:
        net_qty_value = float(qty_match.group(1))
        unit_str = qty_match.group(2)
        if 'fl' in unit_str or 'fluid' in unit_str: unit = 'fl_oz'
        elif 'oz' in unit_str or 'ounce' in unit_str: unit = 'ounce'
        elif 'g' in unit_str or 'gram' in unit_str: unit = 'gram'
        elif 'lb' in unit_str or 'pound' in unit_str: unit = 'pound'
        elif 'l' in unit_str or 'liter' in unit_str: unit = 'liter'
        elif 'ml' in unit_str or 'milliliter' in unit_str: unit = 'ml'
    return num_of_packs, net_qty_value, unit

def extract_item_and_brand(text):
    item_name_match = re.search(r"Item Name:\s*(.*)", text)
    if item_name_match:
        item_name = item_name_match.group(1).strip()
        brand = item_name.split(' ')[0]
        return item_name, brand
    return "Unknown", "Unknown"

# --- Step 1.4: Execution ---
print("\n--- Running Initial Parsing ---")
df_train = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
df_train['catalog_content'] = df_train['catalog_content'].fillna('')
df_train[['item_name', 'brand']] = df_train['catalog_content'].apply(lambda x: pd.Series(extract_item_and_brand(x)))
df_train[['num_of_packs', 'net_qty', 'unit']] = df_train['catalog_content'].apply(lambda x: pd.Series(parse_product_details_v5(x)))
df_train.to_csv(TRAIN_PARSED_PATH, index=False)

print(f"\n✅ Initial parsing complete. Parsed data saved to:\n{TRAIN_PARSED_PATH}")
display(df_train.head())

In [3]:
# ====================================================================
# 🔹 BLOCK 2: IMAGE EMBEDDING GENERATION (RESTART-PROOF)
# ====================================================================

print("\n--- Generating Image Embeddings ---")

# Load the parsed data
df_parsed = pd.read_csv(TRAIN_PARSED_PATH)

# Define the image embedding function
def get_image_embedding(image_url, model, preprocess, device):
    try:
        response = requests.get(image_url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        batch_t = torch.unsqueeze(preprocess(img), 0).to(device)
        with torch.no_grad():
            embedding = model(batch_t)
        return embedding.cpu().numpy().flatten()
    except Exception:
        return np.zeros(576)

df_img_embeds = pd.DataFrame()
if os.path.exists(IMG_EMBED_CHECKPOINT_PATH) and len(pd.read_csv(IMG_EMBED_CHECKPOINT_PATH)) == len(df_parsed):
    print("✅ Full image embedding file found. Skipping.")
else:
    print("⚠️ Full image embedding file not found. Generating embeddings...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    img_model = models.mobilenet_v3_small(pretrained=True)
    img_model.classifier = torch.nn.Identity()
    img_model.to(device); img_model.eval()
    preprocess = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    processed_ids = set()
    if os.path.exists(IMG_EMBED_CHECKPOINT_PATH):
        print("   - Partial checkpoint found. Resuming...")
        df_img_embeds = pd.read_csv(IMG_EMBED_CHECKPOINT_PATH)
        processed_ids = set(df_img_embeds['sample_id'])

    df_to_process = df_parsed[~df_parsed['sample_id'].isin(processed_ids)]

    if not df_to_process.empty:
        new_embeddings = []
        # Process in chunks and save periodically
        SAVE_INTERVAL = 1000
        for i, (index, row) in enumerate(tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Generating Image Embeddings")):
            embedding = get_image_embedding(row['image_link'], img_model, preprocess, device)
            result = {'sample_id': row['sample_id'], **{f'img_embed_{j}': val for j, val in enumerate(embedding)}}
            new_embeddings.append(result)

            if (i + 1) % SAVE_INTERVAL == 0 or (i + 1) == len(df_to_process):
                df_new_chunk = pd.DataFrame(new_embeddings)
                df_img_embeds = pd.concat([df_img_embeds, df_new_chunk], ignore_index=True)
                df_img_embeds.to_csv(IMG_EMBED_CHECKPOINT_PATH, index=False)
                new_embeddings = [] # Clear the batch
                print(f"\n💾 Image embedding checkpoint saved! Processed {len(df_img_embeds)} rows.")

    print("\n✅ Image embedding generation complete.")


--- Generating Image Embeddings ---
✅ Full image embedding file found. Skipping.


In [7]:
# ====================================================================
# 🔹 BLOCK 3: TEXT EMBEDDING GENERATION (RESTART-PROOF)
# ====================================================================

print("\n--- Generating Text Embeddings ---")

# Load the parsed data
df_parsed = pd.read_csv(TRAIN_PARSED_PATH)

df_text_embeds = pd.DataFrame()
if os.path.exists(TEXT_EMBED_CHECKPOINT_PATH) and len(pd.read_csv(TEXT_EMBED_CHECKPOINT_PATH)) == len(df_parsed):
    print("✅ Full text embedding file found. Skipping.")
else:
    print("⚠️ Full text embedding file not found. Generating embeddings...")
    text_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

    processed_ids = set()
    if os.path.exists(TEXT_EMBED_CHECKPOINT_PATH):
        print("   - Partial checkpoint found. Resuming...")
        df_text_embeds = pd.read_csv(TEXT_EMBED_CHECKPOINT_PATH)
        processed_ids = set(df_text_embeds['sample_id'])

    df_to_process = df_parsed[~df_parsed['sample_id'].isin(processed_ids)]

    if not df_to_process.empty:
        texts_to_encode = df_to_process['catalog_content'].fillna('').tolist()
        text_embeddings = text_model.encode(texts_to_encode, show_progress_bar=True, batch_size=128)

        df_new_embeds = pd.DataFrame(text_embeddings, columns=[f'text_embed_{i}' for i in range(text_embeddings.shape[1])])
        df_new_embeds['sample_id'] = df_to_process['sample_id'].values

        df_text_embeds = pd.concat([df_text_embeds, df_new_embeds], ignore_index=True)
        df_text_embeds.to_csv(TEXT_EMBED_CHECKPOINT_PATH, index=False)
        print("\n✅ Text embedding generation complete and saved.")
    else:
        print("   - All text embeddings were already processed.")


--- Generating Text Embeddings ---
✅ Full text embedding file found. Skipping.


In [8]:
# ====================================================================
# 🔹 BLOCK 4: FINAL ASSEMBLY, CLEANING, AND SAVING
# ====================================================================
print("\n--- Assembling all features and performing final cleaning ---")

# Load all the components
df_parsed = pd.read_csv(TRAIN_PARSED_PATH)
df_img_embeds = pd.read_csv(IMG_EMBED_CHECKPOINT_PATH)
df_text_embeds = pd.read_csv(TEXT_EMBED_CHECKPOINT_PATH)

# Merge everything
df_final = pd.merge(df_parsed, df_img_embeds, on='sample_id', how='left')
df_final = pd.merge(df_final, df_text_embeds, on='sample_id', how='left')

df_final['total_qty'] = df_final['num_of_packs'] * df_final['net_qty']
df_final['is_qty_missing'] = ((df_final['num_of_packs'] == 1) & (df_final['net_qty'] == 1.0)).astype(int)

# --- Outlier Capping ---
outlier_caps = {'num_of_packs': 750.00, 'net_qty': 750.00, 'total_qty': 12000.00}
for col, cap_value in outlier_caps.items():
    if col in df_final.columns:
        df_final.loc[df_final[col] > cap_value, col] = cap_value
print("✅ Outliers capped.")

# --- Correct Scaling (Bug Free) ---
num_cols_to_scale = ['num_of_packs', 'net_qty', 'total_qty']
scaler = StandardScaler()
df_final[num_cols_to_scale] = scaler.fit_transform(df_final[num_cols_to_scale])
print(f"✅ Scaling complete.")

# --- Save final files ---
df_final.to_csv(TRAIN_FINAL_CLEANED_PATH, index=False)
joblib.dump(scaler, SCALER_PATH)
np.save(OUTLIER_CAPS_PATH, outlier_caps)
print(f"\n💾 Final clean data '{TRAIN_FINAL_CLEANED_PATH}' and learned objects saved successfully!")

print("\n\n🎉🎉🎉 SUCCESS! Your entire preprocessing pipeline is complete. 🎉🎉🎉")


--- Assembling all features and performing final cleaning ---
✅ Outliers capped.
✅ Scaling complete.

💾 Final clean data '/content/drive/My Drive/ML_Price_Prediction/dataset/train_final_cleaned_v2.csv' and learned objects saved successfully!


🎉🎉🎉 SUCCESS! Your entire preprocessing pipeline is complete. 🎉🎉🎉
